<a href="https://colab.research.google.com/github/2estherblaise-max/ECON3916-Statical-Machine-Learning./blob/main/LAB16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wbgapi scikit-learn matplotlib pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import wbgapi as wb

np.random.seed(42)
print("Setup complete ✓")

Setup complete ✓


In [ ]:
# ============================================================
# PART 1A: Download World Bank Data
# ============================================================

# WDI indicator codes for our predictors
# Format: 'indicator_code': 'human_readable_name'
INDICATORS = {
    # Growth outcome (our y)
    'NY.GDP.PCAP.KD.ZG': 'gdp_growth_pc',

    # Trade & Openness
    'NE.TRD.GNFS.ZS':    'trade_pct_gdp',
    'BX.KLT.DINV.WD.GD.ZS': 'fdi_inflows_pct_gdp',
    'TM.TAX.MRCH.SM.AR.ZS': 'tariff_rate_avg',
    'BX.GSR.ROYL.CD':    'royalties_receipts',

    # Macroeconomics
    'FP.CPI.TOTL.ZG':    'inflation_cpi',
    'GC.DOD.TOTL.GD.ZS': 'govt_debt_pct_gdp',
    'GC.XPN.TOTL.GD.ZS': 'govt_expenditure_pct_gdp',
    'BN.CAB.XOKA.GD.ZS': 'current_account_pct_gdp',
    'FR.INR.RINR':       'real_interest_rate',
    'PA.NUS.FCRF':       'exchange_rate_official',

    # Education & Human Capital
    'SE.SEC.ENRR':       'secondary_enrollment_gross',
    'SE.TER.ENRR':       'tertiary_enrollment_gross',
    'SE.ADT.LITR.ZS':    'adult_literacy_rate',
    'SE.XPD.TOTL.GD.ZS': 'education_expenditure_pct_gdp',
    'SL.UEM.TOTL.ZS':    'unemployment_rate',

    # Infrastructure & Technology
    'IT.NET.USER.ZS':    'internet_users_pct',
    'IT.CEL.SETS.P2':    'mobile_subscriptions_per100',
    'EG.ELC.ACCS.ZS':    'electricity_access_pct',
    'IS.ROD.PAVE.ZS':    'paved_roads_pct',

    # Health & Demographics
    'SP.DYN.LE00.IN':    'life_expectancy',
    'SH.DYN.MORT':       'infant_mortality_per1000',
    'SP.POP.GROW':       'population_growth',
    'SP.URB.TOTL.IN.ZS': 'urbanization_pct',
    'SH.XPD.CHEX.GD.ZS': 'health_expenditure_pct_gdp',

    # Finance & Banking
    'FS.AST.DOMS.GD.ZS': 'domestic_credit_pct_gdp',
    'CM.MKT.LCAP.GD.ZS': 'market_cap_pct_gdp',
    'FB.ATM.TOTL.P5':    'atms_per100k',
    'FD.AST.PRVT.GD.ZS': 'private_credit_pct_gdp',

    # Natural Resources
    'NY.GDP.TOTL.RT.ZS': 'natural_resource_rents_pct_gdp',
    'EG.FEC.RNEW.ZS':    'renewable_energy_pct',
    'EN.ATM.CO2E.PC':    'co2_emissions_per_capita',

    # Agriculture
    'NV.AGR.TOTL.ZS':    'agriculture_pct_gdp',
    'AG.LND.ARBL.ZS':    'arable_land_pct',

    # Governance (World Bank Governance Indicators)
    'IQ.CPA.TRAD.XQ':    'trade_cpia',
    'IQ.CPA.FINS.XQ':    'financial_management_cpia',
    'IQ.CPA.PROP.XQ':    'property_rights_cpia',
}

OUTCOME_VAR = 'gdp_growth_pc'
indicator_list = list(INDICATORS.keys())

print(f"Downloading {len(indicator_list)} indicators for all countries, 2013–2019...")
print("(This may take 30–60 seconds — API call to World Bank)")

# Download 7 years of data (2013–2019, pre-COVID) and average
# wb.data.DataFrame returns a DataFrame indexed by (economy, time) or (economy)
try:
    raw_data = wb.data.DataFrame(
        indicator_list,
        time=range(2013, 2020),  # 2013–2019
        skipBlanks=True,
        labels=False
    )
    raw_data.columns = [INDICATORS[c] if c in INDICATORS else c for c in raw_data.columns]
    print(f"Raw data shape: {raw_data.shape}")
    print("Download successful ✓")
except Exception as e:
    print(f"API error: {e}")
    print("Loading fallback data from CSV...")
    # Fallback: load pre-downloaded CSV
    # raw_data = pd.read_csv('data/fallback_wdi_topic16.csv', index_col=[0, 1])

(This may take 30–60 seconds — API call to World Bank)
Raw data shape: (7255, 7)
Download successful ✓


In [ ]:
dfs = []
for code, name in INDICATORS.items():
    try:
        temp = wb.data.DataFrame(code, time=range(2013, 2020), labels=False)
        temp = temp.mean(axis=1).rename(name)  # average across years → one value per country
        dfs.append(temp)
    except:
        pass

country_data = pd.concat(dfs, axis=1)
print(f"Downloaded: {country_data.shape}")
print(f"Columns: {list(country_data.columns[:5])}...")

# Drop countries with < 60% non-missing
threshold = 0.60
country_data = country_data.dropna(thresh=int(threshold * country_data.shape[1]))

# Drop indicators with < 60% non-missing
country_data = country_data.dropna(axis=1, thresh=int(threshold * len(country_data)))

# Impute remaining with median
country_data = country_data.fillna(country_data.median())

print(f"\nFinal dataset: {len(country_data)} countries × {country_data.shape[1]} indicators")
print(f"Sample countries: {list(country_data.index[:5])}")
print(f"Indicators retained: {list(country_data.columns)}")
print(f"\nGDP growth summary:")
print(country_data[OUTCOME_VAR].describe().round(2))

Downloaded: (266, 35)
Columns: ['gdp_growth_pc', 'trade_pct_gdp', 'fdi_inflows_pct_gdp', 'tariff_rate_avg', 'royalties_receipts']...

Final dataset: 238 countries × 29 indicators
Sample countries: ['ABW', 'AFE', 'AFG', 'AFW', 'AGO']
Indicators retained: ['gdp_growth_pc', 'trade_pct_gdp', 'fdi_inflows_pct_gdp', 'tariff_rate_avg', 'royalties_receipts', 'inflation_cpi', 'govt_expenditure_pct_gdp', 'current_account_pct_gdp', 'real_interest_rate', 'exchange_rate_official', 'secondary_enrollment_gross', 'tertiary_enrollment_gross', 'adult_literacy_rate', 'education_expenditure_pct_gdp', 'unemployment_rate', 'internet_users_pct', 'mobile_subscriptions_per100', 'electricity_access_pct', 'life_expectancy', 'infant_mortality_per1000', 'population_growth', 'urbanization_pct', 'health_expenditure_pct_gdp', 'atms_per100k', 'private_credit_pct_gdp', 'natural_resource_rents_pct_gdp', 'renewable_energy_pct', 'agriculture_pct_gdp', 'arable_land_pct']

GDP growth summary:
count    238.00
mean       1.76

In [ ]:
# ============================================================
# PART 1C: Train-Test Split & Feature Standardization
# ============================================================

feature_cols = [c for c in country_data.columns if c != OUTCOME_VAR]

X = country_data[feature_cols].values
y = country_data[OUTCOME_VAR].values
feature_names = feature_cols

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print(f"Training countries: {X_train.shape[0]}")
print(f"Test countries:     {X_test.shape[0]}")
print(f"Number of predictors: {X_train.shape[1]}")
print(f"p/n ratio: {X_train.shape[1]}/{X_train.shape[0]} = {X_train.shape[1]/X_train.shape[0]:.2f}")
print()
print("If p/n > 0.5, OLS is at serious risk of overfitting.")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\nFeatures standardized ✓")

Training countries: 166
Test countries:     72
Number of predictors: 28
p/n ratio: 28/166 = 0.17

If p/n > 0.5, OLS is at serious risk of overfitting.

Features standardized ✓


In [ ]:
# ============================================================
# PART 1D: OLS Baseline — Demonstrating the Failure Mode
# ============================================================

ols_model = LinearRegression()
ols_model.fit(X_train_scaled, y_train)

y_train_pred_ols = ols_model.predict(X_train_scaled)
y_test_pred_ols  = ols_model.predict(X_test_scaled)

ols_train_r2 = r2_score(y_train, y_train_pred_ols)
ols_test_r2  = r2_score(y_test,  y_test_pred_ols)

print("=" * 45)
print("OLS BASELINE RESULTS")
print("=" * 45)
print(f"Training R²:  {ols_train_r2:.3f}")
print(f"Test R²:      {ols_test_r2:.3f}")
print(f"Gap:          {ols_train_r2 - ols_test_r2:.3f}  ← overfitting signal")
print("=" * 45)
print()
if ols_train_r2 - ols_test_r2 > 0.3:
    print("⚠️  Large gap confirmed — OLS is overfitting badly.")
else:
    print("Gap is moderate — but regularization will still improve generalization.")

OLS BASELINE RESULTS
Training R²:  0.600
Test R²:      -0.849
Gap:          1.449  ← overfitting signal

⚠️  Large gap confirmed — OLS is overfitting badly.


In [ ]:
lambda_grid = np.logspace(-2, 3, 50)

ridge_cv = RidgeCV(alphas=lambda_grid, cv=5)
ridge_cv.fit(X_train_scaled, y_train)

y_test_pred_ridge = ridge_cv.predict(X_test_scaled)
ridge_test_r2  = r2_score(y_test, y_test_pred_ridge)
ridge_test_mse = mean_squared_error(y_test, y_test_pred_ridge)

print("=" * 45)
print("RIDGE REGRESSION RESULTS")
print("=" * 45)
print(f"Optimal λ* (CV-selected): {ridge_cv.alpha_:.4f}")
print(f"Non-zero coefficients:    {np.sum(ridge_cv.coef_ != 0)} of {X_train.shape[1]}")
print(f"Test R²:                  {ridge_test_r2:.3f}")
print(f"Test MSE:                 {ridge_test_mse:.3f}")
print(f"\nvs. OLS: Test R² = {ols_test_r2:.3f}")

RIDGE REGRESSION RESULTS
Optimal λ* (CV-selected): 47.1487
Non-zero coefficients:    28 of 28
Test R²:                  -0.051
Test MSE:                 4.691

vs. OLS: Test R² = -0.849
